<p align="center">

<h1> Team Members </h1>

<b>Yasmin Sherif Eid</b><br>
<i>ID: 20242394</i>
<br>
<b>Menna Tullah Mahmoud Abd ElMoneim</b><br>
<i>ID: 20242351</i> 

</p>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer

<h2>Task 1</h2>

In [2]:
np.random.seed(42)

def train_test_split(X, y, test_size=0.2, seed=42):
    rng = np.random.RandomState(seed)
    train_idx, test_idx = [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y)

print("Full:\t", np.bincount(y), np.round(np.bincount(y)/len(y), 4))
print("Train:\t", np.bincount(y_train), np.round(np.bincount(y_train)/len(y_train), 4))
print("Test:\t", np.bincount(y_test), np.round(np.bincount(y_test)/len(y_test), 4))


Full:	 [212 357] [0.3726 0.6274]
Train:	 [170 286] [0.3728 0.6272]
Test:	 [42 71] [0.3717 0.6283]


<h2>Task 2</h2>

In [ ]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        k, d = len(self.classes_), X.shape[1]      

        self.theta_ = np.zeros((k, d))     
        self.var_ = np.zeros((k, d))        
        self.class_prior_ = np.zeros(k)   
        self.epsilon_ = self.var_smoothing * np.var(X, axis=0).max()

        for i, c in enumerate(self.classes_):
            X_c = X[y==c]
            self.theta_[i] = X_c.mean(axis=0)
            self.var_[i] = X_c.var(axis=0) + self.epsilon_
            self.class_prior_[i] = X_c.shape[0] / X.shape[0]

        return self

    def _joint_log_likelihood(self, X):
        X = np.asarray(X, dtype=float)
        jill = np.zeros((X.shape[0], len(self.classes_)))
        for i in range(len(self.classes_)):
            log_prior = np.log(self.class_prior_[i])
            log_like =  -0.5 * np.sum(
                np.log(2 * np.pi * self.var_[i]) + (X - self.theta_[i]) ** 2 / self.var_[i], axis=1
            )
            jill[:, i] = log_prior + log_like

        return jill

    def predict(self, X):
        return self.classes_[np.argmax(self._joint_log_likelihood(X), axis=1)]

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        m = jll.max(axis=1, keepdims=True)
        log_norm = m + np.log(np.sum(np.exp(jll - m), axis=1, keepdims=True))
        return jll - log_norm

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))